In [2]:
from allophant.estimator import Estimator
import torch
from torch.nn.utils.rnn import pad_sequence
import torchaudio
from allophant.dataset_processing import Batch
from datasets import load_from_disk
from allophant import predictions
import pandas as pd
from pathlib import Path
from IPython.display import Audio
import os
import json
import soundfile as sf
from transformers import pipeline
from tqdm import tqdm

In [3]:
device = torch.device(0)
model, attribute_indexer = Estimator.restore("kgnlp/allophant", device=device)
supported_features = attribute_indexer.feature_names
# The phonetic feature categories supported by the model, including "phonemes"
print(supported_features)

/home/mjsimmons/projects/allophant/allophant/phonetic_features.py:1114: LanguageMappingWarning: Remapped some languages to a variant within the same macro language: {'swa': 'swh', 'est': 'ekk'}
  warnings.warn(
/home/mjsimmons/projects/allophant/allophant/phonetic_features.py:1184: SingletonFeatureWarning: Only one feature variant found in ['tone']
  warnings.warn(
/home/mjsimmons/.env/.allophant/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


['stress', 'syllabic', 'short', 'long', 'consonantal', 'sonorant', 'continuant', 'delayedRelease', 'approximant', 'tap', 'trill', 'nasal', 'lateral', 'labial', 'round', 'labiodental', 'coronal', 'anterior', 'distributed', 'strident', 'dorsal', 'high', 'low', 'front', 'back', 'tense', 'retractedTongueRoot', 'advancedTongueRoot', 'periodicGlottalSource', 'epilaryngealSource', 'spreadGlottis', 'constrictedGlottis', 'fortis', 'raisedLarynxEjective', 'loweredLarynxImplosive', 'click', 'phoneme']


In [4]:
DENTAL_T = 't̪'
DENTAL_D = 'd̪'
IPA_G = 'ɡ'

TIRA_STOPS = [
    'p', DENTAL_T, 't', 'c', 'k', 'ʔ',
    'b', DENTAL_D, 'd', 'ɟ', IPA_G,
]
TIRA_FRICATIVES = [
    'f', 's', 'ʃ','h',
    'v', 'ð',
]
TIRA_GLIDES = [
    'w', 'j',
]
TIRA_NASALS = [
    'm', 'n', 'ɲ', 'ŋ',
]
TIRA_SONORANTS = [
    'l', 'r', 'ɾ', #'ɽ', # allophant doesn't support retroflex flap
]
TIRA_VOWELS = [
    'i',      'u',
    'ɪ',      'ʊ',
    'e', 'ə', 'o',
    'ɛ', 'ɜ', 'ɔ',
         'a',
]

TIRA_CONSONANTS = TIRA_STOPS + TIRA_FRICATIVES + TIRA_GLIDES + TIRA_NASALS + TIRA_SONORANTS

inventory = TIRA_CONSONANTS + TIRA_VOWELS
inventory_indexer = attribute_indexer.attributes.subset(inventory)
inventory[:10], len(inventory)

(['p', 't̪', 't', 'c', 'k', 'ʔ', 'b', 'd̪', 'd', 'ɟ'], 37)

In [5]:
dataset_dir = Path(os.environ['DATASETS'])
train_manifest = dataset_dir/'tira_asr_translated'/'nemo'/'nemo_train_manifest.jsonl'
val_manifest = dataset_dir/'tira_asr_translated'/'nemo'/'nemo_validation_manifest.jsonl'

model_dir = Path(os.environ['MODELS'])
mbart_path = model_dir/'mbart_ft_allophant_condensed'

In [6]:
def load_json_lines(manifest_path):
    data = []
    with open(manifest_path) as f:
        lines = f.readlines()
    for line in lines:
        data.append(json.loads(line))
    return data

In [7]:
val_data = load_json_lines(val_manifest)
val_data[:2]

[{'audio_filepath': '/mnt/LocalStorage/mjsimmons//datasets/tira_asr_translated/nemo/validation/cfebfbf6-6692-42fc-9449-e64d4be49f7f.wav',
  'duration': 1.94,
  'text': 'the shepherd is good',
  'target_lang': 'en',
  'source_text': 'ìjɔ̀ kə̀cə̀lò',
  'source_lang': 'sw',
  'task': 'ast',
  'pnc': 'yes'},
 {'audio_filepath': '/mnt/LocalStorage/mjsimmons//datasets/tira_asr_translated/nemo/validation/67dc4b49-0289-43bd-b487-b265d87eba49.wav',
  'duration': 0.89,
  'text': 'he (the shepherd) is good',
  'target_lang': 'en',
  'source_text': 'kə̀cə̀lò',
  'source_lang': 'sw',
  'task': 'ast',
  'pnc': 'yes'}]

In [8]:
def transcribe_allophant(audio_path):
    audio, sr = sf.read(audio_path)
    audio = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
    lengths = torch.tensor([audio.shape[-1],])
    lang_id = torch.zeros(1)

    batch = Batch(audio, lengths, lang_id)
    model_outputs = model.predict(
      batch.to(device),
      attribute_indexer.composition_feature_matrix(inventory).to(device)
    )
    # Create a feature mapping for your inventory and CTC decoders for the desired feature set
    ctc_decoders = predictions.feature_decoders(inventory_indexer, feature_names=supported_features)
    decoder = ctc_decoders['phoneme']
    decoded = decoder(model_outputs.outputs['phoneme'].transpose(1, 0), model_outputs.lengths)
    phonemes = []
    for [hypothesis] in decoded:
        # NOTE: token indices are offset by one due to the <BLANK> token used during decoding
        recognized = inventory_indexer.feature_values('phoneme', hypothesis.tokens - 1)
        recognized = ''.join(recognized)
        phonemes.append(recognized)
    return phonemes[0]

In [9]:
mbart_pipeline = pipeline(
    task="translation",
    model=mbart_path,
    device=0,
    src_lang="sw_KE",
    tgt_lang="en_XX",
)

In [10]:
def get_mbart_allophant_output(pipeline, data, i):
    src = data[i]['source_text']
    allophant_src = transcribe_allophant(data[i]['audio_filepath'])
    reference = data[i]['text']
    pred = mbart_pipeline(allophant_src, max_length=200)[0]['translation_text']

    return src, allophant_src, reference, pred

def print_mbart_allophant_output(pipeline, data, i):
    print("mBART-50 w Allophant...")
    src, allophant_src, reference, pred = get_mbart_allophant_output(pipeline, data, i)

    print("\n" + "="*40)
    print(f"REF:\t\t{reference}")
    print(f"PRED:\t\t{pred}")
    print(f"SRC:\t\t{src}")
    print(f"ALLOPHANT:\t{allophant_src}")
    print("="*40)
    return Audio(data[i]['audio_filepath'])

In [24]:
# i = 95
# i = 713
# i = 842
# i = 592
# i = 213
# i = 250
# i = 118

i = 713
print_mbart_allophant_output(mbart_pipeline, val_data, i)

mBART-50 w Allophant...

REF:		Kuku will make the boy drink
PRED:		Kuku drank water
SRC:		kúkù kə̀t̪ìjí ápɾíɲà
ALLOPHANT:	kukokt̪iaprea


In [12]:
# rows = []
# for i, _ in enumerate(tqdm(val_data)):
#     src, allophant_src, reference, pred = get_mbart_allophant_output(mbart_pipeline, val_data, i)
#     rows.append({
#         'src': src,
#         'allophant_src': allophant_src,
#         'ref': reference,
#         'mBART_machine_label_pred': pred,
#     })
# df = pd.DataFrame(rows)
# df.to_csv('mBART_machine_labels.csv')
    